In [ ]:
!pip install rdkit
!pip install torch
!pip install torchani
# pip install rdkit-pypi torch torchani (if using ANI)
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np

SMILES = "N[C@@H](C)C(=O)O"  # L-alanine (neutral example). For zwitterion, protonate/deprotonate appropriately.
NAME = "alanine_neutral"
NCONF = 150                 # more for larger side chains
RMS_PRUNE = 0.4             # Å
KEEP_MMFF = 50              # keep this many lowest by MMFF energy
USE_ANI = True             # set True to compute ANI-2x energies here

def prepare_mol(smiles):
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)
    return m

def embed_minimize_confs(mol, nconf=NCONF):
    params = AllChem.ETKDGv3()
    params.pruneRmsThresh = -1.0   # we’ll prune ourselves
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=nconf, params=params)
    # MMFF minimize
    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94s')
    e_list = []
    for cid in conf_ids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            e = 1e9
        e_list.append((cid, e))
    return e_list

def prune_by_rmsd(mol, conf_ids, rms_cut=RMS_PRUNE):
    kept = []
    for cid in conf_ids:
        keep = True
        for kc in kept:
            rms = rdMolAlign.GetBestRMS(mol, mol, prbId=cid, refId=kc)
            if rms < rms_cut:
                keep = False
                break
        if keep:
            kept.append(cid)
    return kept

def mmff_rank_and_prune(mol, conf_energy_pairs, keep=KEEP_MMFF):
    conf_energy_pairs = sorted(conf_energy_pairs, key=lambda x: x[1])
    # Take top 'keep' by energy but ensure diversity with RMSD pruning
    ranked = [cid for cid,_ in conf_energy_pairs]
    diverse = prune_by_rmsd(mol, ranked, rms_cut=RMS_PRUNE)
    # keep the best among those diverse; if too many, cap at 'keep'
    diverse_sorted = sorted(diverse, key=lambda cid: dict(conf_energy_pairs)[cid])
    return diverse_sorted[:keep]

def compute_ani_energies(mol, conf_ids, model_name='ani2x'):
    import torch, torchani
    # Load model
    model = (torchani.models.ANI2x() if model_name.lower() == 'ani2x'
             else torchani.models.ANI1ccx())
    device = torch.device('cpu')
    model = model.to(device).eval()

    # Map atomic numbers -> element symbols for TorchANI
    z2sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F', 16:'S', 17:'Cl', 35:'Br', 53:'I'}
    symbols = [z2sym[atom.GetAtomicNum()] for atom in mol.GetAtoms()]

    # TorchANI helper to build species tensor
    species = model.consts.species_to_tensor(symbols).unsqueeze(0).to(device)  # shape (1, natoms)

    energies = {}
    for cid in conf_ids:
        conf = mol.GetConformer(cid)
        coords = [[conf.GetAtomPosition(i).x,
                   conf.GetAtomPosition(i).y,
                   conf.GetAtomPosition(i).z] for i in range(mol.GetNumAtoms())]
        coordinates = torch.tensor([coords], dtype=torch.float32, device=device)  # (1, natoms, 3)
        with torch.no_grad():
            e = model((species, coordinates)).energies.item()  # Hartree
        energies[cid] = e
    return energies  # dict: confId -> Eh


def write_sdf(mol, conf_ids, fields, path):
    w = Chem.SDWriter(path)
    for cid in conf_ids:
        m = Chem.Mol(mol)
        m.SetProp("_Name", f"{NAME}_conf{cid}")
        for k,v in fields.items():
            if cid in v:
                m.SetDoubleProp(k, float(v[cid]))
        w.write(m, confId=cid)
    w.close()

# === 1) Prepare input ===
mol0 = prepare_mol(SMILES)

# (Optional) you’d generate protomers externally (e.g., Dimorphite at pH 7) and loop each through the same steps.
tauts = [mol0]

print(f"Found {len(tauts)} unique tautomers")
best_overall = None  # (energy_Eh, taut_idx, conf_id)

for i, taut in enumerate(tauts):
    confEs = embed_minimize_confs(taut, NCONF)
    keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)

    aniE = {}
    if USE_ANI:
        aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')

    # Report for this tautomer
    if USE_ANI and aniE:
        cid_min = min(aniE, key=lambda k: aniE[k])
        E_min = aniE[cid_min]   # Hartree
        print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
        if (best_overall is None) or (E_min < best_overall[0]):
            best_overall = (E_min, i, cid_min)
    else:
        # fall back to MMFF (NOT electronic) just so something prints
        mmffE = dict(confEs)
        cid_min = min(keep_ids, key=lambda k: mmffE[k])
        print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")

if USE_ANI and best_overall:
    E, ti, ci = best_overall
    print(f"\nGround-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")


  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 5.0 MB/s eta 0:00:0000:0100:02m
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.19.1-py3-none-any.whl (15 kB)
Using cached fsspec-2025.9.0-py3-none-any.whl (199 kB)
Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torch]32m5/6 [torch]]x]
  Using cached torchani-2.2.4-py3-none-any.whl.metadata (6.0 kB)
  Using cached lark_parser-0.12.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached torchani-2.2.4-py3-none-any.whl (10.9 MB)
Using cached lark_parser-0.12.0-py2.py3-none-any.whl (103

/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/aev.py:16: UserWarning: cuaev not installed
  warnings.warn("cuaev not installed")
/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/__init__.py:55: UserWarning: Dependency not satisfied, torchani.ase will not be available
  warnings.warn("Dependency not satisfied, torchani.ase will not be available")


/Users/sanskriti/miniconda3/envs/fusion1/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -323.661033044 Eh  (conf 2)

Ground-state estimate (ANI): -323.661033044 Eh  from tautomer 0, conformer 2
